# F1 Aero GEM-CNN Training on Google Colab

This notebook allows you to train the F1 Aero model on a Google Colab GPU (T4, L4, or A100).

### Setup Instructions:
1. **Runtime Type:** Ensure you are using a GPU runtime (Runtime -> Change runtime type -> T4 GPU).
2. **Upload Project:** Upload your `f1_aero_gem` folder to your Google Drive.
3. **Data:** Either upload your data to Drive or run the download cell below.

In [ ]:
# @title 1. Check GPU and Environment
import torch
gpu_available = torch.cuda.is_available()
print(f"GPU Available: {gpu_available}")
if gpu_available:
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Go to Runtime > Change runtime type and select T4 GPU.")

In [ ]:
# @title 2. Mount Google Drive and Copy Project
from google.colab import drive
import os
import shutil

try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f'Mount failed: {e}')
    print('TIP: Use an Incognito window or log out of other Google accounts.')

# Update this path to where you uploaded the project folder in Drive
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/f1_aero_gem'
LOCAL_PROJECT_PATH = '/content/f1_aero_gem'

if os.path.exists(DRIVE_PROJECT_PATH):
    print("Copying project from Drive to local storage for faster access...")
    if os.path.exists(LOCAL_PROJECT_PATH):
        shutil.rmtree(LOCAL_PROJECT_PATH)
    shutil.copytree(DRIVE_PROJECT_PATH, LOCAL_PROJECT_PATH)
    os.chdir(LOCAL_PROJECT_PATH)
    print(f"Current working directory: {os.getcwd()}")
else:
    print(f"ERROR: Project folder not found at {DRIVE_PROJECT_PATH}")

In [ ]:
# @title 3. Install Dependencies
import torch
torch_ver = torch.__version__.split('+')[0]
cuda_ver = 'cu121' # Standard for Colab T4

print(f"Installing PyG for PyTorch {torch_ver} and CUDA {cuda_ver}...")

!pip install -q torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html
!pip install -q pyvista vtk pyyaml tqdm requests scipy

print("Installation complete.")

In [ ]:
# @title 4. Prepare Data (Optional if already uploaded)
# Run this if you need to download the DrivAerNet++ dataset
import os

DOWNLOAD_DATA = False # @param {type:"boolean"}

if DOWNLOAD_DATA:
    print("Downloading dataset (top 500 designs)...")
    !python download_drivarnet.py
    !python prepare_data.py
else:
    print("Skipping download. Ensure data is present in your project folder.")

In [ ]:
# @title 5. Update Config for Colab Paths
import yaml
import os

config_path = 'configs/f1_base.yaml'
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# Point to the local data directory if it exists locally, 
# otherwise keep what is in the config (which might be the Mac path)
local_data_path = os.path.join(os.getcwd(), 'data', 'drivaernet_real')
if os.path.exists(local_data_path):
    cfg['data']['data_root'] = local_data_path
    print(f"Updating data_root to: {local_data_path}")

with open(config_path, 'w') as f:
    yaml.dump(cfg, f)

print("Config updated for Colab environment.")

In [ ]:
# @title 6. Run Training
!python -m train.trainer --config configs/f1_base.yaml

In [ ]:
# @title 7. Sync Results back to Drive
import shutil

RESULTS_DIR = '/content/f1_aero_gem/runs'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/f1_aero_gem/runs_colab'

if os.path.exists(RESULTS_DIR):
    print(f"Syncing results to {DRIVE_RESULTS_DIR}...")
    if os.path.exists(DRIVE_RESULTS_DIR):
        shutil.rmtree(DRIVE_RESULTS_DIR)
    shutil.copytree(RESULTS_DIR, DRIVE_RESULTS_DIR)
    print("Results synced successfully.")
else:
    print("No results found to sync.")